In [ ]:
# Upload the dataset
from google.colab import files

uploaded = files.upload()

In [ ]:
# Import the required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score, recall_score
from tabulate import tabulate
from scipy.stats import ks_2samp, wasserstein_distance

In [ ]:
# INPUT SELECTION
SEED = 42

MODEL_TAG = 'cwgan'          # ctgan_perclass | cwgan | tablegan | medgan
DEVICE_TAG = 'gpu'
EPOCHS = 500                     # 100 | 300 | 500
TARGET_PER_CLASS = 6000

STEM = f'{MODEL_TAG}_{DEVICE_TAG}_epoch{EPOCHS}_augment{TARGET_PER_CLASS}_seed{SEED}'

COMBINED_FILE = f'{STEM}.csv'
SYNTHETIC_ONLY_FILE = f'{STEM}_synthetic_only.csv'

TRAIN_DATASET = 'df_train.csv'
df_train = pd.read_csv(TRAIN_DATASET)

# The KS test, the Wasserstein distance and the KDE plots must compare
SYNTHETIC_DATASET = SYNTHETIC_ONLY_FILE
df_synthetic = pd.read_csv(SYNTHETIC_DATASET)

print('Real training data :', TRAIN_DATASET, df_train.shape)
print('Synthetic data     :', SYNTHETIC_DATASET, df_synthetic.shape)

_overlap = (set(map(tuple, df_train.round(6).to_numpy()))
            & set(map(tuple, df_synthetic.round(6).to_numpy())))
print('Rows appearing in both :', len(_overlap))
if _overlap:
    print('WARNING - the synthetic file still contains real records.')
    print('          Check that the _synthetic_only file was uploaded.')

In [ ]:
pd.set_option('display.max_columns', None)
df_train.head()

In [ ]:
pd.set_option('display.max_columns', None)
df_synthetic.head()

In [ ]:
selected_cols = [
    'Age',
    'Neurological Assessments',
    'Weight Gain During Pregnancy',
    'Blood Glucose Levels',
    'Insulin Levels',
    'Digestive Enzyme Levels',
    'Cholesterol Levels',
    'Blood Pressure',
    'BMI',
    'Waist Circumference',
    'Birth Weight',
    'Pancreatic Health',
    'Pulmonary Function'

]

## Balanced sampling


In [ ]:
# BALANCED SAMPLING FOR THE COMPARISON
label_col = 'Target'

counts_train = df_train[label_col].value_counts()
counts_synth = df_synthetic[label_col].value_counts()

print("Real data      : {} classes, smallest {}, largest {}"
      .format(len(counts_train), counts_train.min(), counts_train.max()))
print("Synthetic data : {} classes, smallest {}, largest {}"
      .format(len(counts_synth), counts_synth.min(), counts_synth.max()))

missing = set(counts_train.index) ^ set(counts_synth.index)
if missing:
    print("\nWARNING - these classes appear in only one of the two sets:", sorted(missing))

SAMPLES_PER_CLASS = int(min(counts_train.min(), counts_synth.min()))
print("\nSamples per class used for both sets :", SAMPLES_PER_CLASS)
print("Total compared per set               :",
      SAMPLES_PER_CLASS * len(counts_train))

df_train_sampled = (
    df_train
    .groupby(label_col, group_keys=False)
    .apply(lambda x: x.sample(n=SAMPLES_PER_CLASS, random_state=42))
)

df_synthetic_sampled = (
    df_synthetic
    .groupby(label_col, group_keys=False)
    .apply(lambda x: x.sample(n=SAMPLES_PER_CLASS, random_state=42))
)

assert len(df_train_sampled) == len(df_synthetic_sampled), \
    "The two samples must be the same size."

print("\nReal sample      :", df_train_sampled.shape)
print("Synthetic sample :", df_synthetic_sampled.shape)

## Distribution comparison: real vs synthetic


In [ ]:
sns.set(style="whitegrid")

# Number of panels per row
cols_per_row = 4

for i in range(0, len(selected_cols), cols_per_row):
    subset_cols = selected_cols[i:i + cols_per_row]

    fig, axes = plt.subplots(
        1, len(subset_cols),
        figsize=(16, 3.5),
        sharey=False
    )

    if len(subset_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, subset_cols):
        if col in df_train_sampled.columns and col in df_synthetic_sampled.columns:
            sns.kdeplot(
                df_train_sampled[col],
                fill=True,
                color='blue',
                alpha=0.4,
                linewidth=1.5,
                label='Real Data',
                ax=ax
            )

            sns.kdeplot(
                df_synthetic_sampled[col],
                fill=True,
                color='orange',
                alpha=0.4,
                linewidth=1.5,
                linestyle='--',
                label='Synthetic Data',
                ax=ax
            )

            ax.set_title(col, fontsize=10)
            ax.set_xlabel('')
            ax.set_ylabel('Density')

    axes[0].legend(fontsize=9)
    plt.tight_layout()
    plt.show()

## Kolmogorov-Smirnov test and Wasserstein distance



In [ ]:
results = []
for col in selected_cols:
    if col in df_train_sampled.columns and col in df_synthetic_sampled.columns:
        train_data = df_train_sampled[col].dropna()
        ctgan_data = df_synthetic_sampled[col].dropna()
        # KS-test
        ks_stat, p_value = ks_2samp(train_data, ctgan_data)
        # Wasserstein distance
        w_dist = wasserstein_distance(train_data, ctgan_data)
        results.append({
            'Feature': col,
            'KS Statistic': ks_stat,
            'p-value': p_value,
            'Wasserstein Distance': w_dist
        })

df_stats = pd.DataFrame(results)

# Sort from the best (smallest KS statistic) to the worst
df_stats = df_stats.sort_values('KS Statistic', ascending=True).reset_index(drop=True)

df_stats